# CS781 Phase 7 — Prior Case Retrieval

Your Phase 1 backbones → relevance training → split-specific full-pool retrieval → verified top seven.
Follows §13 of `Capstone Document_edited.pdf` and the professor's starter progression: authentication,
configuration, data, model/adaptation, training, validation, saved predictions and ZIP.

All **six official files** are used. Train query relevance supervises fitting; dev selects checkpoints
and ranking policies; test supplies queries and its own candidate pool only. No external pretrained
retriever, embedding model or LLM API is used. BM25 is a provided-text lexical baseline.

Primary: **Micro-F1@7**, secondary: **Micro-Precision@7 and Micro-Recall@7**. The full gold set is
never truncated/padded to seven. Output is an object mapping each original query ID to **seven unique
original candidate ID strings**, ordered by predicted relevance and drawn only from that split's pool.

Six core profiles cover all four Phase 1 origins, compulsory LoRMA and smaller-student KD, plus full,
partial, prompt and IA³ adaptation. Eight optional profiles and twelve bounded training trials broaden
comparisons. Actual scores must come from runs; more experiments do not guarantee improvement.

## 1. Install dependencies

In [ ]:
%pip -q install "tokenizers>=0.20,<0.24" "huggingface_hub>=0.26,<3" "numpy>=1.26" "scipy>=1.11" "scikit-learn>=1.4" "tqdm>=4.66" matplotlib

## 2. Mount Drive, authenticate and check GPU

In [ ]:
import os, json, math, random, time, zipfile, hashlib, gc, copy, getpass, platform
from pathlib import Path
from contextlib import nullcontext
from collections import Counter
from bisect import bisect_left, bisect_right
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from tokenizers import ByteLevelBPETokenizer

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive')
else:
    BASE_DIR = Path.cwd()

PHASE1_DIR = BASE_DIR / 'cs781_phase1' if IN_COLAB else BASE_DIR
PROJECT_DIR = BASE_DIR / 'cs781_phase7'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(PROJECT_DIR / 'hf_cache')
from huggingface_hub import hf_hub_download, login

HF_TOKEN = os.environ.get('HF_TOKEN')
if IN_COLAB and not HF_TOKEN:
    try:
        HF_TOKEN = userdata.get('HF')
    except Exception:
        pass
def validate_hf_token(value):
    # Never print or include the credential in an error message.
    if not isinstance(value, str):
        raise TypeError('HF token must be a plain text string, not ' + type(value).__name__ + '.')
    value = value.strip()
    if not value.startswith('hf_'):
        raise ValueError('Paste only the Hugging Face personal access token starting with hf_.')
    return value

if HF_TOKEN:
    try:
        HF_TOKEN = validate_hf_token(HF_TOKEN)
    except (TypeError, ValueError) as error:
        print(str(error))
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste only your hf_ token (hidden): ')
HF_TOKEN = validate_hf_token(HF_TOKEN)
login(token=HF_TOKEN, add_to_git_credential=False)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16
print('Device:', DEVICE, '| PyTorch:', torch.__version__)
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('CPU is suitable for checks; select a Colab GPU before full training.')
print('Phase 1 input:', PHASE1_DIR)
print('Phase 7 output:', PROJECT_DIR)

# Atomic writes avoid leaving partially written JSON/checkpoints after interruption.
def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.' + __import__('uuid').uuid4().hex + '.tmp')
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temporary, path)

def save_torch(data, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.' + __import__('uuid').uuid4().hex + '.tmp')
    torch.save(data, temporary)
    os.replace(temporary, path)

def amp_context():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if DEVICE.type == 'cuda' else nullcontext()

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)
import shutil

## 3. All experiment and tuning controls — edit here

Start with `encoder_full`, `RUN_ALL=False`. After verifying the first result, `RUN_ALL=True` runs the
six core profiles, reusing completed matching runs. KD runs last and needs a larger completed Phase 7
teacher. Use a new `RUN_TAG` for changed manual settings. Section 12 contains the controls guide.

In [ ]:
SEED = 42
EXPERIMENT = 'encoder_full'
RUN_ALL = False
RUN_EXTRA_APPROACHES = False
RUN_TUNING = False
RUN_TAG = 'v1'
RESUME = True
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
EPOCHS = 6
QUERY_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
POSITIVES_PER_QUERY = 2
NEGATIVES_PER_QUERY = 4
HARD_NEGATIVE_FRACTION = 0.75
MINE_DENSE_NEGATIVES = False  # Optional train-only mining after each completed epoch.
USE_IN_BATCH_NEGATIVES = True
CHUNK_LENGTH = 256
CHUNK_OVERLAP = 48
MAX_CHUNKS = 3  # 0 = all windows; expensive for long candidate pools.
CHUNK_BATCH_SIZE = 8
ENCODE_DOCUMENT_BATCH_SIZE = 8
CHUNK_POLICY = 'spread'  # spread, head, tail, head_tail
POOLING = 'attention'  # attention, mean, max, mean_max
EMBEDDING_DIM = 256
HEAD_DROPOUT = 0.1
SHARE_QUERY_CANDIDATE_PROJECTION = True
LOSS_NAME = 'multi_positive'  # multi_positive (uniform positives), pairwise, combined
TEMPERATURE = 0.05
PAIRWISE_MARGIN = 0.1
LEARNING_RATE = None
HEAD_LR_MULTIPLIER = 3.0
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1
EARLY_STOPPING_PATIENCE = 3
GRADIENT_CHECKPOINTING = True
RANK = 8
ALPHA = 16.0
PROMPT_TOKENS = 16
PARTIAL_LAYERS = 2
KD_WEIGHT = 0.5
KD_TEMPERATURE = 2.0
TEACHER_RUN_DIR = None  # Completed, compatible, larger Phase 7 retrieval teacher.
SAVE_EVERY_STEPS = 50
MAX_OPTIMIZER_STEPS = None
BM25_K1 = 1.5
BM25_B = 0.75
BM25_MAX_FEATURES = 60000
BM25_STOP_WORDS = 'english'  # None preserves stop words.
HARD_NEGATIVE_DEPTH = 128
RRF_K = 60
RRF_DEPTH = 200
# All selected trained-model outputs retain a nonzero neural contribution.
DENSE_WEIGHTS = [1.0, 0.75, 0.5, 0.25]
SEARCH_LATE_INTERACTION = True
ENABLE_CITATION_FEATURE = False  # Exact provided-document IndianKanoon URL matches; no external lookup.
CITATION_WEIGHTS = [0.0, 0.01, 0.03]
EXCLUDE_SELF_MATCH = False  # PDF does not prohibit self IDs; enable only after inspecting gold/pools.
PHASE1_HEADS = {'decoder_only': 12, 'encoder_decoder': 8, 'hybrid_prefixlm': 10, 'lightweight': 12}
TOP_K = 7  # Submission contract; not a tunable hyperparameter.
CORE_EXPERIMENTS = {
    'encoder_full': {'architecture': 'encoder_decoder', 'method': 'full', 'lr': 2e-5},
    'decoder_partial': {'architecture': 'decoder_only', 'method': 'partial', 'lr': 5e-5},
    'hybrid_lorma': {'architecture': 'hybrid_prefixlm', 'method': 'lorma', 'lr': 2e-4},
    'lightweight_prompt': {'architecture': 'lightweight', 'method': 'prompt', 'lr': 2e-3},
    'lightweight_ia3': {'architecture': 'lightweight', 'method': 'ia3', 'lr': 2e-4},
    'lightweight_kd': {'architecture': 'lightweight', 'method': 'kd', 'lr': 2e-5},
}
EXTRA_EXPERIMENTS = {
    'decoder_full': {'architecture': 'decoder_only', 'method': 'full', 'lr': 2e-5},
    'encoder_partial': {'architecture': 'encoder_decoder', 'method': 'partial', 'lr': 5e-5},
    'encoder_lorma': {'architecture': 'encoder_decoder', 'method': 'lorma', 'lr': 2e-4},
    'encoder_ia3': {'architecture': 'encoder_decoder', 'method': 'ia3', 'lr': 2e-4},
    'decoder_ia3': {'architecture': 'decoder_only', 'method': 'ia3', 'lr': 2e-4},
    'hybrid_full': {'architecture': 'hybrid_prefixlm', 'method': 'full', 'lr': 2e-5},
    'lightweight_full': {'architecture': 'lightweight', 'method': 'full', 'lr': 2e-5},
    'lightweight_lorma': {'architecture': 'lightweight', 'method': 'lorma', 'lr': 2e-4},
}
EXPERIMENTS = {**CORE_EXPERIMENTS, **EXTRA_EXPERIMENTS}
TUNE_BASE_EXPERIMENT = 'hybrid_lorma'
TUNING_TRIALS = [
    {'name': 'lr_low', 'overrides': {'lr': 1e-4}},
    {'name': 'more_chunks', 'overrides': {'max_chunks': 6}},
    {'name': 'longer_chunks', 'overrides': {'chunk_length': 384}},
    {'name': 'mean_pooling', 'overrides': {'pooling': 'mean'}},
    {'name': 'mean_max', 'overrides': {'pooling': 'mean_max'}},
    {'name': 'head_tail', 'overrides': {'chunk_policy': 'head_tail'}},
    {'name': 'pairwise', 'overrides': {'loss_name': 'pairwise'}},
    {'name': 'combined_loss', 'overrides': {'loss_name': 'combined'}},
    {'name': 'more_negatives', 'overrides': {'negatives': 8}},
    {'name': 'asymmetric_heads', 'overrides': {'share_projection': False}},
    {'name': 'dense_mining', 'overrides': {'mine_dense_negatives': True}},
    {'name': 'warmer_scores', 'overrides': {'temperature': 0.1}},
]
assert EXPERIMENT in EXPERIMENTS and TUNE_BASE_EXPERIMENT in EXPERIMENTS
assert TOP_K == 7 and EPOCHS > 0 and QUERY_BATCH_SIZE > 0 and GRAD_ACCUM_STEPS > 0
assert MAX_OPTIMIZER_STEPS is None or MAX_OPTIMIZER_STEPS > 0
assert all(0 < value <= 1 for value in DENSE_WEIGHTS)
assert 0 <= HARD_NEGATIVE_FRACTION <= 1 and 0 <= KD_WEIGHT <= 1 and KD_TEMPERATURE > 0
seed_all(SEED)

## 4. Download six files, preserve IDs, audit relevance and build token caches

Candidate pools remain separate, including when the same ID occurs in several pools. Sentence-list
texts are joined in order. Original leading zeros are preserved; an integer gold ID can resolve to a
zero-padded pool ID only if the numeric match is unambiguous. Every gold ID must belong to its pool.
Test relevance fields are ignored. Train queries with no positives are audited and excluded from
positive-supervised batching; all dev queries, including empty gold sets, remain in evaluation.

In [ ]:
def parse_case_file(path):
    import ast, csv, io
    path=Path(path)
    with path.open('rb') as stream:
        signature=stream.read(4)
    encoding='utf-16' if signature.startswith((b'\xff\xfe',b'\xfe\xff')) else 'utf-8-sig'
    try:
        content=path.read_text(encoding=encoding)
    except UnicodeError as exc:
        raise ValueError('File is not UTF-8/UTF-16 text; possibly an incomplete or non-text download.') from exc
    stripped=content.strip()
    if not stripped:
        raise ValueError('Downloaded file is empty or whitespace-only.')
    if stripped.startswith('version https://git-lfs.github.com/spec/v1'):
        raise ValueError('Downloaded file is a Git LFS pointer, not the dataset contents.')
    if stripped.lower().startswith(('<!doctype html','<html','<?xml')):
        raise ValueError('Downloaded file is an HTML/XML response, not case data.')
    try:
        return json.loads(content), 'json'
    except json.JSONDecodeError as exc:
        json_error=f'JSON parse failed at line {exc.lineno}, column {exc.colno}.'
    # Newline-delimited objects are accepted only if every nonblank line is valid.
    lines=io.StringIO(content)
    first=next((line.strip() for line in lines if line.strip()),'')
    if first.startswith('{'):
        try:
            initial=json.loads(first)
            if isinstance(initial,dict):
                rows=[initial]
                for number,line in enumerate(lines,2):
                    if not line.strip(): continue
                    row=json.loads(line)
                    if not isinstance(row,dict): raise ValueError('JSON Lines must contain objects.')
                    rows.append(row)
                return rows,'jsonl'
        except (json.JSONDecodeError,ValueError):
            pass
    # Some exports are CSV despite their .json suffix. Require known columns.
    csv_reader=csv.reader(io.StringIO(content))
    try:
        header=next(csv_reader)
    except (StopIteration,csv.Error):
        header=[]
    if 'id' in header and ('document' in header or 'text' in header):
        csv.field_size_limit(min(__import__('sys').maxsize,2**31-1))
        reader=csv.DictReader(io.StringIO(content),strict=True)
        rows=[]
        try:
            for row in reader:
                if None in row or any(value is None for value in row.values()):
                    raise ValueError('CSV row has missing or extra columns.')
                rows.append(row)
        except csv.Error as exc:
            raise ValueError('Malformed CSV case export.') from exc
        return rows,'csv'
    # Safe parsing only; never execute downloaded Python code.
    if stripped.lstrip('[').lstrip().startswith(('{'+chr(39),'{"')):
        try:
            literal=ast.literal_eval(stripped)
            if isinstance(literal,(list,dict)):
                return literal,'python_literal'
        except (SyntaxError,ValueError,TypeError,RecursionError):
            pass
    raise ValueError(json_error+' Unrecognized or malformed case-file format; no records were skipped.')



FILES={
 'train_queries':'Phase_7_PCR/Train/train.json',
 'train_candidates':'Phase_7_PCR/Train/train_candidates.json',
 'dev_queries':'Phase_7_PCR/Dev/dev.json',
 'dev_candidates':'Phase_7_PCR/Dev/dev_candidates.json',
 'test_queries':'Phase_7_PCR/Test/test.json',
 'test_candidates':'Phase_7_PCR/Test/test_candidates.json'}
DOWNLOAD_REPORT={}
def load_download(key):
    for attempt in range(2):
        path=Path(hf_hub_download(repo_id=HF_DATASET_REPO,filename=FILES[key],repo_type='dataset',token=HF_TOKEN,force_download=attempt==1))
        print('Reading',key,path.stat().st_size,'bytes')
        try:
            raw,kind=parse_case_file(path)
            DOWNLOAD_REPORT[key]={'format':kind,'bytes':path.stat().st_size,'attempt':attempt+1}
            save_json(DOWNLOAD_REPORT,PROJECT_DIR/'dataset_loading_report.json')
            return raw
        except ValueError as error:
            DOWNLOAD_REPORT[key]={'error':str(error),'attempt':attempt+1}
            save_json(DOWNLOAD_REPORT,PROJECT_DIR/'dataset_loading_report.json')
            if attempt: raise ValueError(f'{key}: parsing failed after a fresh download: {error}') from error
            print('Retrying with fresh download:',key,error)

def case_key(value):
    if isinstance(value, bool) or not isinstance(value, (str, int)):
        raise ValueError('Case IDs must be strings or integers; original leading zeros are preserved.')
    return str(value)


def text_value(value):
    if value is None: return ''
    if isinstance(value, str): return value
    if isinstance(value, dict): value = list(value.values())
    if isinstance(value, list):
        if all(isinstance(item, str) for item in value): return '\n'.join(value)
        if all(isinstance(item, dict) and isinstance(item.get('text'), str) for item in value):
            return '\n'.join(item['text'] for item in value)
    raise ValueError('Expected case text or an ordered list of sentence strings/text objects.')


def normalize_records(raw, queries=False, labeled=False):
    if isinstance(raw, dict) and 'id' in raw and 'text' in raw: entries = [(None, raw)]
    elif isinstance(raw, dict) and isinstance(raw.get('data'), list): entries = [(None, row) for row in raw['data']]
    elif isinstance(raw, dict): entries = list(raw.items())
    elif isinstance(raw, list): entries = [(None, row) for row in raw]
    else: raise ValueError('Expected case objects or an ID-keyed mapping.')
    result = {}
    for outer, row in entries:
        if not queries and isinstance(row, (str, list)): row = {'id': outer, 'text': row}
        if not isinstance(row, dict): raise ValueError('Invalid case object.')
        original = row.get('id', outer); key = case_key(original)
        if outer is not None and case_key(outer) != key: raise ValueError('Embedded ID differs from mapping key.')
        text = text_value(row.get('text', row.get('document')))
        relevant = None
        if queries and labeled:
            relevant = row.get('relevant_candidates')
            if isinstance(relevant, str):
                try: relevant = json.loads(relevant)
                except json.JSONDecodeError:
                    try: relevant = __import__('ast').literal_eval(relevant)
                    except (ValueError, SyntaxError): raise ValueError(f'{key}: relevant_candidates is not a valid list.')
            if not isinstance(relevant, list): raise ValueError(f'{key}: relevant_candidates must be a list on train/dev queries.')
            relevant = list(dict.fromkeys(case_key(value['id'] if isinstance(value, dict) else value) for value in relevant))
        normalized = {'id': key, 'original_id': original, 'text': text, 'relevant': relevant}
        if key in result:
            if result[key] != normalized: raise ValueError(f'Conflicting duplicate case ID {key}; no rows were overwritten.')
            continue
        result[key] = normalized
    return result


QUERIES = {}; CANDIDATES = {}; DATA_AUDIT = {}
for split in ['train', 'dev', 'test']:
    QUERIES[split] = normalize_records(load_download(split + '_queries'), True, split != 'test')
    CANDIDATES[split] = normalize_records(load_download(split + '_candidates'))
    if not QUERIES[split] or len(CANDIDATES[split]) < TOP_K:
        raise ValueError(f'{split}: need queries and at least {TOP_K} candidate IDs.')
    numeric = {}
    for candidate in CANDIDATES[split]:
        if candidate.isdigit(): numeric.setdefault(str(int(candidate)), []).append(candidate)
    conversions = 0
    for query, row in QUERIES[split].items():
        if split == 'test':
            assert row['relevant'] is None
            continue
        resolved = []
        for candidate in row['relevant']:
            if candidate not in CANDIDATES[split]:
                options = numeric.get(str(int(candidate)), []) if candidate.isdigit() else []
                if len(options) != 1: raise ValueError(f'{split}/{query}: gold ID {candidate} is absent/ambiguous in this split candidate pool.')
                candidate = options[0]; conversions += 1
            resolved.append(candidate)
        row['relevant'] = list(dict.fromkeys(resolved))
        if EXCLUDE_SELF_MATCH and query in row['relevant']: raise ValueError('Gold includes self ID; disable EXCLUDE_SELF_MATCH.')
    if EXCLUDE_SELF_MATCH and any(len(CANDIDATES[split]) - (query in CANDIDATES[split]) < TOP_K for query in QUERIES[split]):
        raise ValueError('Self exclusion leaves fewer than seven candidates.')
    DATA_AUDIT[split] = {'queries': len(QUERIES[split]), 'candidates': len(CANDIDATES[split]),
                         'empty_query_texts': sum(not row['text'].strip() for row in QUERIES[split].values()),
                         'empty_candidate_texts': sum(not row['text'].strip() for row in CANDIDATES[split].values()),
                         'unambiguous_gold_id_conversions': conversions,
                         'queries_without_gold': sum(not row['relevant'] for row in QUERIES[split].values()) if split != 'test' else None}
    if all(not row['text'].strip() for row in CANDIDATES[split].values()): raise ValueError(f'{split}: all candidate texts are empty; inspect source schema.')
    if all(not row['text'].strip() for row in QUERIES[split].values()): raise ValueError(f'{split}: all query texts are empty; inspect source schema.')
    print(split, DATA_AUDIT[split])
TRAIN_KEYS = [key for key, row in QUERIES['train'].items() if row['relevant']]
TRAIN_KEY_SET = set(TRAIN_KEYS)
if not TRAIN_KEYS: raise ValueError('No training queries have positive candidates.')
fingerprint = hashlib.sha256(str(EXCLUDE_SELF_MATCH).encode())
for split in ['train', 'dev', 'test']:
    for kind, records in [('queries', QUERIES[split]), ('candidates', CANDIDATES[split])]:
        for key, row in records.items(): fingerprint.update(json.dumps([split, kind, key, row], sort_keys=True, ensure_ascii=False).encode())
DATA_FINGERPRINT = fingerprint.hexdigest()
DATA_SELF_EXCLUSION = EXCLUDE_SELF_MATCH
save_json({'fingerprint': DATA_FINGERPRINT, 'files': FILES, 'splits': DATA_AUDIT,
           'train_queries_with_positive_supervision': TRAIN_KEYS,
           'train_queries_without_positive_supervision': [key for key in QUERIES['train'] if key not in TRAIN_KEY_SET]}, PROJECT_DIR / 'dataset_manifest.json')

TOKENIZER_DIR = PHASE1_DIR / 'bpe_tokenizer'
tok = ByteLevelBPETokenizer(str(TOKENIZER_DIR / 'vocab.json'), str(TOKENIZER_DIR / 'merges.txt'))
VOCAB_SIZE_ACTUAL = tok.get_vocab_size()
PAD_ID = tok.token_to_id('<pad>'); BOS_ID = tok.token_to_id('<bos>'); EOS_ID = tok.token_to_id('<eos>'); UNK_ID = tok.token_to_id('<unk>')
if any(value is None for value in [PAD_ID, BOS_ID, EOS_ID, UNK_ID]): raise ValueError('Missing Phase 1 BPE special tokens.')
TOKENIZER_HASH = hashlib.sha256((TOKENIZER_DIR / 'vocab.json').read_bytes() + (TOKENIZER_DIR / 'merges.txt').read_bytes()).hexdigest()


class TokenStore:
    def __init__(self, path, index):
        self.data = np.memmap(path, mode='r', dtype='<i4') if path.stat().st_size else np.empty(0, dtype='<i4'); self.index = index
    def __getitem__(self, key):
        start, length = self.index[key]; return self.data[start:start + length].tolist()


TOKENS = {}
for split in ['train', 'dev', 'test']:
    TOKENS[split] = {}
    for kind, records in [('queries', QUERIES[split]), ('candidates', CANDIDATES[split])]:
        stem = f'tokens_{split}_{kind}_{DATA_FINGERPRINT[:16]}_{TOKENIZER_HASH[:12]}'
        path = PROJECT_DIR / (stem + '.bin'); index_path = PROJECT_DIR / (stem + '.json')
        if not path.is_file() or not index_path.is_file():
            temporary = path.with_name(path.name + '.' + __import__('uuid').uuid4().hex + '.tmp'); index = {}; offset = 0
            with temporary.open('wb') as stream:
                for key, row in tqdm(records.items(), desc='Tokenizing ' + split + '/' + kind, unit='case'):
                    ids = np.asarray(tok.encode(row['text'], add_special_tokens=False).ids, dtype='<i4')
                    stream.write(ids.tobytes()); index[key] = [offset, len(ids)]; offset += len(ids)
            os.replace(temporary, path); save_json(index, index_path)
        index = json.loads(index_path.read_text())
        if set(index) != set(records): raise ValueError('Token cache IDs disagree.')
        active = path
        if IN_COLAB:
            staging = Path('/content/cs781_phase7_tokens'); staging.mkdir(exist_ok=True); active = staging / path.name
            if not active.is_file() or active.stat().st_size != path.stat().st_size:
                temporary = active.with_name(active.name + '.' + __import__('uuid').uuid4().hex + '.tmp')
                with path.open('rb') as source, temporary.open('wb') as target:
                    with tqdm.wrapattr(source, 'read', total=path.stat().st_size, desc='Staging ' + split + '/' + kind, bytes=True) as progress:
                        shutil.copyfileobj(progress, target, 8 * 1024 * 1024)
                os.replace(temporary, active)
        TOKENS[split][kind] = TokenStore(active, index)
length_report = {}
for split in TOKENS:
    length_report[split] = {}
    for role, store in TOKENS[split].items():
        lengths = np.asarray([length for offset, length in store.index.values()])
        length_report[split][role] = {'cases': len(lengths), 'tokens_p50': float(np.median(lengths)),
                                     'tokens_p90': float(np.quantile(lengths, 0.9)), 'tokens_max': int(lengths.max())}
save_json(length_report, PROJECT_DIR / 'document_lengths.json')
print('Token length audit:', length_report)

## 5. Exact @7 metrics and split-specific BM25 baseline/hard negatives

BM25 is fitted separately on each candidate pool without relevance labels. The held-out baseline
helps determine whether learned representations add useful ranking information. It does not count
as a Phase 1 adaptation experiment. Hard negatives always come from TRAIN and exclude every known
positive, not just the sampled positive. Metrics count the complete gold relevance set.

In [ ]:
def validate_predictions(predictions, split):
    if not isinstance(predictions, dict) or set(predictions) != set(QUERIES[split]):
        raise ValueError('Predictions must contain every query ID exactly once.')
    for query, values in predictions.items():
        if not isinstance(values, list) or len(values) != TOP_K:
            raise ValueError(f'{query}: expected exactly seven unique candidate IDs.')
        if any(type(value) is not str or value not in CANDIDATES[split] for value in values):
            raise ValueError(f'{query}: candidate IDs must be original strings from the {split} candidate pool.')
        if len(set(values)) != TOP_K: raise ValueError(f'{query}: candidate IDs must be unique.')
        if EXCLUDE_SELF_MATCH and query in values: raise ValueError('Self candidate was excluded by configuration.')
    return True


def retrieval_metrics(predictions, split='dev'):
    if split == 'test': raise ValueError('Public test gold is private; no test score can be computed.')
    validate_predictions(predictions, split)
    tp = 0; gold_count = 0; per_query = {}; ceiling = 0
    for query, row in QUERIES[split].items():
        gold = set(row['relevant']); selected = set(predictions[query][:TOP_K]); hits = len(gold & selected)
        tp += hits; gold_count += len(gold); ceiling += min(TOP_K, len(gold))
        per_query[query] = {'hits_at_7': hits, 'gold_count': len(gold), 'precision_at_7': hits / TOP_K,
                            'recall_at_7': hits / len(gold) if gold else 0.0,
                            'missed': sorted(gold - selected), 'irrelevant_retrieved': sorted(selected - gold)}
    predicted_count = len(predictions) * TOP_K; denominator = predicted_count + gold_count
    return {'micro_f1_at_7': 2 * tp / denominator if denominator else 0.0,
            'micro_precision_at_7': tp / predicted_count if predicted_count else 0.0,
            'micro_recall_at_7': tp / gold_count if gold_count else 0.0,
            'true_positives': tp, 'predicted_count': predicted_count, 'gold_count': gold_count,
            'queries': len(predictions), 'oracle_micro_f1_at_7': 2 * ceiling / denominator if denominator else 0.0,
            'per_query': per_query}


def show_scores(metrics, title):
    print(title)
    print(f"Micro-F1@7 [PRIMARY]: {metrics['micro_f1_at_7']:.6f}; Micro-Precision@7: {metrics['micro_precision_at_7']:.6f}; Micro-Recall@7: {metrics['micro_recall_at_7']:.6f}")
    print('Queries:', metrics['queries'], '; hits:', metrics['true_positives'], '; complete gold count:', metrics['gold_count'])


def save_submission(output, predictions):
    validate_predictions(predictions, 'test'); save_json(predictions, output / 'prediction.json')
    temporary = output / 'submission.zip.tmp'
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
        archive.write(output / 'prediction.json', arcname='prediction.json')
    os.replace(temporary, output / 'submission.zip')
    with zipfile.ZipFile(output / 'submission.zip') as archive:
        assert archive.namelist() == ['prediction.json']
        saved = json.loads(archive.read('prediction.json')); validate_predictions(saved, 'test'); assert saved == predictions


def stable_order(scores, ids):
    values = np.asarray(scores, dtype=np.float64)
    if len(values) != len(ids) or not np.isfinite(values).all(): raise ValueError('Nonfinite or misaligned ranking scores.')
    return np.lexsort((np.asarray(ids, dtype=str), -values))


def allowed_mask(query, ids):
    return np.asarray([not EXCLUDE_SELF_MATCH or candidate != query for candidate in ids])


def top_predictions(scores, ids, query):
    order = stable_order(scores, ids)
    return [ids[index] for index in order if not EXCLUDE_SELF_MATCH or ids[index] != query][:TOP_K]

In [ ]:
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer


class BM25Index:
    def __init__(self, split):
        self.ids = list(CANDIDATES[split]); self.position = {key: i for i, key in enumerate(self.ids)}
        self.vectorizer = CountVectorizer(max_features=BM25_MAX_FEATURES, stop_words=BM25_STOP_WORDS,
            token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z0-9_]{1,}\b|\b\d+\b', dtype=np.float32)
        texts = [CANDIDATES[split][key]['text'] for key in self.ids]
        try: counts = self.vectorizer.fit_transform(texts).tocsr()
        except ValueError as error: raise ValueError(f'{split}: empty lexical vocabulary; inspect candidate texts/stop-word settings.') from error
        lengths = np.asarray(counts.sum(1)).ravel(); mean = max(float(lengths.mean()), 1e-8)
        df = np.asarray((counts > 0).sum(0)).ravel()
        idf = np.log1p((len(self.ids) - df + 0.5) / (df + 0.5))
        weights = counts.tocoo(copy=True)
        tf = weights.data.copy()
        weights.data = (tf * (BM25_K1 + 1) / (tf + BM25_K1 * (1 - BM25_B + BM25_B * lengths[weights.row] / mean))) * idf[weights.col]
        self.weights = weights.tocsr()
    def scores(self, text):
        query = self.vectorizer.transform([text]).tocsr()
        # Presence-only query terms avoid long-query repeated boilerplate dominating scores.
        query.data.fill(1.0)
        return (query @ self.weights.T).toarray().ravel().astype(np.float64)


BM25 = {}; LEXICAL_TOP = {}; BM25_BASELINE = {}; BM25_RANKS = {}
LEXICAL_SIGNATURE = {'k1': BM25_K1, 'b': BM25_B, 'max_features': BM25_MAX_FEATURES,
                     'stop_words': BM25_STOP_WORDS, 'hard_depth': HARD_NEGATIVE_DEPTH}
for split in ['train', 'dev', 'test']:
    print('Building split-specific BM25:', split)
    BM25[split] = BM25Index(split); LEXICAL_TOP[split] = {}; BM25_RANKS[split] = {}; predictions = {}
    for query, row in tqdm(QUERIES[split].items(), desc='BM25 ranking ' + split, unit='query'):
        scores = BM25[split].scores(row['text']); order = stable_order(scores, BM25[split].ids)
        BM25_RANKS[split][query] = order[:max(HARD_NEGATIVE_DEPTH, RRF_DEPTH)].astype(np.int32)
        ranked = [BM25[split].ids[i] for i in order if not EXCLUDE_SELF_MATCH or BM25[split].ids[i] != query]
        LEXICAL_TOP[split][query] = ranked[:HARD_NEGATIVE_DEPTH]
        predictions[query] = ranked[:TOP_K]
    BM25_BASELINE[split] = predictions
baseline_folder = PROJECT_DIR / 'bm25_baseline'; baseline_folder.mkdir(exist_ok=True)
bm25_metrics = retrieval_metrics(BM25_BASELINE['dev'])
save_json(bm25_metrics, baseline_folder / 'metrics.json'); save_json(BM25_BASELINE['dev'], baseline_folder / 'dev_prediction.json')
save_submission(baseline_folder, BM25_BASELINE['test'])
show_scores(bm25_metrics, 'BM25 baseline — no pretrained external model; does not count as a neural adaptation experiment')

## 6. Checkpoint-compatible Phase 1 architectures and retrieval adaptation

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]


def apply_rope(q, k, cos, sin):
    q_ = (q * cos) + (rotate_half(q) * sin)
    k_ = (k * cos) + (rotate_half(k) * sin)
    return q_, k_


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, max_len):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer("mask", torch.tril(torch.ones(max_len, max_len)).bool(), persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~self.mask[:T, :T], float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class GPTBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class DecoderOnlyGPT(nn.Module):
    """Architecture 1: classic decoder-only Transformer LM."""

    def __init__(self, vocab_size, n_layer, n_embd, n_head, d_ff, max_len=2048):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(max_len, n_embd)
        self.blocks = nn.ModuleList([GPTBlock(n_embd, n_head, d_ff, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        # Simplified, non-KV-cached generation (recomputes full forward pass each step).
        # Positions are re-based to 0 each call since context is re-clipped to max_len.
        self.eval()
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            _, logits = self(ids_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

In [ ]:
class RelativePositionBias(nn.Module):
    def __init__(self, n_head, num_buckets=32, max_distance=128, bidirectional=True):
        super().__init__()
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, n_head)

    @staticmethod
    def _bucket(rel_pos, bidirectional, num_buckets, max_distance):
        ret = 0
        n = -rel_pos
        if bidirectional:
            num_buckets //= 2
            ret += (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2
        is_small = n < max_exact
        val_large = max_exact + (
            torch.log(n.float() / max_exact + 1e-6) / math.log(max_distance / max_exact) * (num_buckets - max_exact)
        ).long()
        val_large = torch.clamp(val_large, max=num_buckets - 1)
        ret = ret + torch.where(is_small, n, val_large)
        return ret

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        rel_pos = k_pos - q_pos
        buckets = self._bucket(rel_pos, self.bidirectional, self.num_buckets, self.max_distance)
        values = self.embedding(buckets)                    # (q_len, k_len, n_head)
        return values.permute(2, 0, 1).unsqueeze(0)          # (1, n_head, q_len, k_len)


class RelPosAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x_q, x_kv, bias=None, causal=False, key_padding_mask=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, Tq, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if bias is not None:
            att = att + bias
        if causal:
            causal_mask = torch.tril(torch.ones(Tq, Tk, device=x_q.device)).bool()
            att = att.masked_fill(~causal_mask, float("-inf"))
        if key_padding_mask is not None:
            pad_keep = key_padding_mask[:, None, None, :].bool()   # (B,1,1,Tk), 1 = keep
            att = att.masked_fill(~pad_keep, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.o_proj(out)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = RelPosAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, bias, key_padding_mask):
        h = self.ln1(x)
        x = x + self.attn(h, h, bias=bias, causal=False, key_padding_mask=key_padding_mask)
        x = x + self.ff(self.ln2(x))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = RelPosAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = RelPosAttention(d_model, n_head)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, enc_out, self_bias, enc_padding_mask):
        h = self.ln1(x)
        x = x + self.self_attn(h, h, bias=self_bias, causal=True)
        x = x + self.cross_attn(self.ln2(x), enc_out, bias=None, causal=False, key_padding_mask=enc_padding_mask)
        x = x + self.ff(self.ln3(x))
        return x


class EncoderDecoderTransformer(nn.Module):
    """Architecture 2: T5-style encoder-decoder Transformer with relative position bias."""

    def __init__(self, vocab_size, n_enc_layer, n_dec_layer, d_model, n_head, d_ff):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, n_head, d_ff) for _ in range(n_enc_layer)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, n_head, d_ff) for _ in range(n_dec_layer)])
        self.enc_rel_bias = RelativePositionBias(n_head, bidirectional=True)
        self.dec_rel_bias = RelativePositionBias(n_head, bidirectional=False)
        self.ln_enc_f = nn.LayerNorm(d_model)
        self.ln_dec_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def encode(self, src_ids, src_mask):
        x = self.tok_emb(src_ids)
        bias = self.enc_rel_bias(src_ids.size(1), src_ids.size(1), src_ids.device)
        for layer in self.enc_layers:
            x = layer(x, bias, src_mask)
        return self.ln_enc_f(x)

    def forward(self, src_ids, src_mask, tgt_in, tgt_out=None):
        enc_out = self.encode(src_ids, src_mask)
        x = self.tok_emb(tgt_in)
        bias = self.dec_rel_bias(tgt_in.size(1), tgt_in.size(1), tgt_in.device)
        for layer in self.dec_layers:
            x = layer(x, enc_out, bias, src_mask)
        x = self.ln_dec_f(x)
        logits = self.head(x)
        loss = None
        if tgt_out is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1), ignore_index=PAD_ID)
        return loss, logits

    @torch.no_grad()
    def generate(self, src_ids, src_mask, max_new_tokens, bos_id, eos_id, temperature=0.8, top_p=0.95):
        # Simplified, non-KV-cached autoregressive decoding.
        self.eval()
        enc_out = self.encode(src_ids, src_mask)
        B = src_ids.size(0)
        tgt = torch.full((B, 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            x = self.tok_emb(tgt)
            bias = self.dec_rel_bias(tgt.size(1), tgt.size(1), tgt.device)
            h = x
            for layer in self.dec_layers:
                h = layer(h, enc_out, bias, src_mask)
            h = self.ln_dec_f(h)
            logits = self.head(h)[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            tgt = torch.cat([tgt, next_id], dim=1)
        return tgt

In [ ]:
class RoPEAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, attn_mask, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~attn_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class HybridBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = RoPEAttention(d_model, n_head)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff)

    def forward(self, x, attn_mask, cos, sin):
        x = x + self.attn(self.norm1(x), attn_mask, cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


def build_prefix_lm_mask(prompt_len, total_len, device):
    """prompt<->prompt: bidirectional. continuation->prompt: visible.
    continuation<->continuation: causal."""
    mask = torch.zeros(total_len, total_len, dtype=torch.bool, device=device)
    mask[:prompt_len, :prompt_len] = True
    mask[prompt_len:, :prompt_len] = True
    cont_len = total_len - prompt_len
    if cont_len > 0:
        mask[prompt_len:, prompt_len:] = torch.tril(torch.ones(cont_len, cont_len, device=device)).bool()
    return mask


class HybridPrefixLM(nn.Module):
    """Architecture 3: single-stack encoder/hybrid LM (RoPE, RMSNorm, SwiGLU,
    UniLM-style prefix-bidirectional + causal-continuation attention mask)."""

    def __init__(self, vocab_size, n_layer, d_model, n_head, d_ff, max_len=1536):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([HybridBlock(d_model, n_head, d_ff) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, prompt_len, labels=None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        cos, sin = build_rope_cache(T, self.d_model // self.n_head, input_ids.device)
        mask = build_prefix_lm_mask(prompt_len, T, input_ids.device).unsqueeze(0).unsqueeze(0)
        for blk in self.blocks:
            x = blk(x, mask, cos, sin)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, prompt_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        input_ids = prompt_ids
        prompt_len = prompt_ids.size(1)
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            cur_prompt_len = min(prompt_len, ids_cond.size(1))
            _, logits = self(ids_cond, cur_prompt_len)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

In [ ]:
class GQAAttention(nn.Module):
    """Grouped-Query Attention: n_kv_head < n_head, reducing parameters and
    (at inference, with KV caching) memory -- the key efficiency trick here."""

    def __init__(self, d_model, n_head, n_kv_head):
        super().__init__()
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.d_head = d_model // n_head
        self.q_proj = nn.Linear(d_model, n_head * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_head * self.d_head, d_model, bias=False)

    def forward(self, x, causal_mask, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~causal_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


class LightweightBlock(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff)

    def forward(self, x, mask, cos, sin):
        x = x + self.attn(self.norm1(x), mask, cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class LightweightLlama(nn.Module):
    """Architecture 4: efficient/lightweight decoder-only Transformer
    (RoPE, RMSNorm, SwiGLU, Grouped-Query Attention)."""

    def __init__(self, vocab_size, n_layer, d_model, n_head, n_kv_head, d_ff, max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([LightweightBlock(d_model, n_head, n_kv_head, d_ff) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        cos, sin = build_rope_cache(T, self.d_model // self.n_head, input_ids.device)
        mask = torch.tril(torch.ones(T, T, device=input_ids.device)).bool().unsqueeze(0).unsqueeze(0)
        for blk in self.blocks:
            x = blk(x, mask, cos, sin)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            _, logits = self(ids_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

In [ ]:
def torch_load(path):
    return torch.load(path, map_location='cpu', weights_only=True)

def infer_phase1_config(architecture, state):
    vocab, dim = state['tok_emb.weight'].shape
    if vocab != VOCAB_SIZE_ACTUAL:
        raise ValueError(f'Phase 1 vocabulary {vocab} != tokenizer {VOCAB_SIZE_ACTUAL}')
    config = {'vocab_size': vocab}
    def count_layers(prefix):
        return len({int(k.split('.')[1]) for k in state if k.startswith(prefix + '.')})
    heads = PHASE1_HEADS[architecture]
    if architecture == 'decoder_only':
        config.update(n_layer=count_layers('blocks'), n_embd=dim, n_head=heads,
                      d_ff=state['blocks.0.mlp.0.weight'].shape[0], max_len=state['pos_emb.weight'].shape[0])
    elif architecture == 'encoder_decoder':
        stored_heads = state['enc_rel_bias.embedding.weight'].shape[1]
        if heads != stored_heads:
            raise ValueError('Encoder head count differs from stored relative-position bias.')
        config.update(n_enc_layer=count_layers('enc_layers'), n_dec_layer=count_layers('dec_layers'),
                      d_model=dim, n_head=heads, d_ff=state['enc_layers.0.ff.0.weight'].shape[0])
    elif architecture == 'hybrid_prefixlm':
        config.update(n_layer=count_layers('blocks'), d_model=dim, n_head=heads,
                      d_ff=state['blocks.0.ff.w1.weight'].shape[0], max_len=1152)
    else:
        head_dim = dim // heads
        config.update(n_layer=count_layers('blocks'), d_model=dim, n_head=heads,
                      n_kv_head=state['blocks.0.attn.k_proj.weight'].shape[0] // head_dim,
                      d_ff=state['blocks.0.ff.w1.weight'].shape[0], max_len=1024)
    if dim % heads:
        raise ValueError('Invalid Phase 1 attention head count.')
    return config

MODEL_CLASSES = {'decoder_only': DecoderOnlyGPT, 'encoder_decoder': EncoderDecoderTransformer,
                 'hybrid_prefixlm': HybridPrefixLM, 'lightweight': LightweightLlama}

def load_phase1(architecture):
    path = PHASE1_DIR / f'run_{architecture}' / 'final_model.pt'
    if not path.is_file():
        raise FileNotFoundError(f'Missing Phase 1 weights: {path}')
    print('Loading:', path)
    state = torch_load(path)
    config = infer_phase1_config(architecture, state)
    print('Inferred config:', config)
    backbone = MODEL_CLASSES[architecture](**config)
    backbone.load_state_dict(state, strict=True)
    del state
    gc.collect()
    original_parameters = sum(p.numel() for p in backbone.parameters())
    del backbone.head
    if architecture == 'encoder_decoder':
        del backbone.dec_layers, backbone.dec_rel_bias, backbone.ln_dec_f
    return backbone, config, original_parameters

In [ ]:
class ProjectionAdapter(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__(); self.base = base; base.requires_grad_(False)
        self.A = nn.Linear(base.out_features, rank, bias=False); self.B = nn.Linear(rank, base.out_features, bias=False)
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5)); nn.init.zeros_(self.B.weight)
    def forward(self, x):
        wx = F.linear(x, self.base.weight, None)
        y = wx + self.scale * self.B(self.A(wx))
        return y if self.base.bias is None else y + self.base.bias


class IA3Projection(nn.Module):
    def __init__(self, base, input_scale=False, fused_kv=False):
        super().__init__(); self.base = base; base.requires_grad_(False)
        self.input_scale = input_scale; self.fused_kv = fused_kv
        dimension = base.in_features if input_scale else (base.out_features * 2 // 3 if fused_kv else base.out_features)
        self.scale = nn.Parameter(torch.ones(dimension))
    def forward(self, x):
        if self.input_scale: return self.base(x * self.scale)
        y = self.base(x)
        scale = torch.cat([torch.ones(y.shape[-1] // 3, device=y.device, dtype=y.dtype), self.scale]) if self.fused_kv else self.scale
        return y * scale


class RetrievalFeatures(nn.Module):
    def __init__(self, backbone, architecture, cfg):
        super().__init__(); self.backbone = backbone; self.architecture = architecture; self.cfg = cfg
        self.soft_prompt = None
        if cfg['spec']['method'] == 'prompt':
            self.soft_prompt = nn.Parameter(torch.empty(cfg['prompt_tokens'], backbone.tok_emb.embedding_dim))
            nn.init.normal_(self.soft_prompt, std=0.02)
    def layer_call(self, layer, *args):
        if self.training and self.cfg['gradient_checkpointing'] and (args[0].requires_grad or any(p.requires_grad for p in layer.parameters())):
            return checkpoint(layer, *args, use_reentrant=False)
        return layer(*args)
    def forward(self, ids, mask, content):
        b = self.backbone; x = b.tok_emb(ids)
        if self.soft_prompt is not None:
            x = torch.cat([self.soft_prompt[None].expand(len(ids), -1, -1), x], dim=1)
            mask = torch.cat([torch.ones(len(ids), len(self.soft_prompt), dtype=torch.bool, device=ids.device), mask], dim=1)
            content = torch.cat([torch.zeros(len(ids), len(self.soft_prompt), dtype=torch.bool, device=ids.device), content], dim=1)
        length = x.shape[1]
        if self.architecture == 'decoder_only':
            if length > b.max_len: raise ValueError('Chunk/prompt exceeds Phase 1 positions.')
            x = x + b.pos_emb(torch.arange(length, device=ids.device))[None]
            for layer in b.blocks: x = self.layer_call(layer, x)
            x = b.ln_f(x)
        elif self.architecture == 'encoder_decoder':
            bias = b.enc_rel_bias(length, length, ids.device)
            for layer in b.enc_layers: x = self.layer_call(layer, x, bias, mask)
            x = b.ln_enc_f(x)
        else:
            cos, sin = build_rope_cache(length, b.d_model // b.n_head, ids.device)
            keep = mask[:, None, None, :].expand(-1, 1, length, -1)
            if self.architecture == 'lightweight':
                keep = keep & torch.ones(length, length, dtype=torch.bool, device=ids.device).tril()[None, None]
            for layer in b.blocks: x = self.layer_call(layer, x, keep, cos, sin)
            x = b.norm_f(x)
        weight = (mask & content).to(x.dtype)
        mean = (x * weight[:, :, None]).sum(1) / weight.sum(1, keepdim=True).clamp_min(1)
        final = x[torch.arange(len(ids), device=ids.device), mask.sum(1) - 1]
        return torch.cat([mean, final], dim=-1)


class LegalRetriever(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__(); self.cfg = cfg; self.features = RetrievalFeatures(backbone, cfg['spec']['architecture'], cfg)
        dimension = backbone.tok_emb.embedding_dim; out = cfg['embedding_dim']
        self.project = nn.Sequential(nn.LayerNorm(2 * dimension), nn.Linear(2 * dimension, out), nn.GELU(), nn.Dropout(cfg['dropout']))
        self.gate = nn.Linear(out, 1); self.mean_max = nn.Linear(2 * out, out)
        self.query_projection = nn.Linear(out, out, bias=False); self.candidate_projection = nn.Linear(out, out, bias=False)
        with torch.no_grad(): self.query_projection.weight.copy_(torch.eye(out)); self.candidate_projection.weight.copy_(torch.eye(out))
        self.candidate_projection.requires_grad_(not cfg['share_projection'])
        self.gate.requires_grad_(cfg['pooling'] == 'attention')
        self.mean_max.requires_grad_(cfg['pooling'] == 'mean_max')
    def forward(self, batch, role, return_chunks=False):
        features = []
        for start in range(0, len(batch['ids']), self.cfg['chunk_batch_size']):
            stop = start + self.cfg['chunk_batch_size']
            features.append(self.features(batch['ids'][start:stop], batch['mask'][start:stop], batch['content'][start:stop]))
        chunks = self.project(torch.cat(features)); documents = []
        for owner in range(len(batch['keys'])):
            h = chunks[batch['owners'] == owner]
            if self.cfg['pooling'] == 'attention': pooled = (h * torch.softmax(self.gate(h).squeeze(-1), dim=0)[:, None]).sum(0)
            elif self.cfg['pooling'] == 'mean': pooled = h.mean(0)
            elif self.cfg['pooling'] == 'max': pooled = h.max(0).values
            elif self.cfg['pooling'] == 'mean_max': pooled = self.mean_max(torch.cat([h.mean(0), h.max(0).values]))
            else: raise ValueError('Unknown pooling.')
            documents.append(pooled)
        projection = self.query_projection if self.cfg['share_projection'] or role == 'queries' else self.candidate_projection
        vectors = F.normalize(projection(torch.stack(documents)), dim=-1)
        if return_chunks: return vectors, F.normalize(projection(chunks), dim=-1), batch['owners']
        return vectors


def make_model(cfg):
    backbone, phase1, original_parameters = load_phase1(cfg['spec']['architecture'])
    soft = cfg['prompt_tokens'] if cfg['spec']['method'] == 'prompt' else 0
    if cfg['chunk_length'] + soft > phase1.get('max_len', 4096): raise ValueError('Chunk plus soft prompt exceeds Phase 1 positions.')
    method = cfg['spec']['method']; selected = None
    if method == 'kd':
        selected = list(range(0, len(backbone.blocks), 2)); backbone.blocks = nn.ModuleList([backbone.blocks[i] for i in selected])
    backbone.requires_grad_(method in ('full', 'kd'))
    targets = []
    if method == 'partial':
        layers = backbone.enc_layers if cfg['spec']['architecture'] == 'encoder_decoder' else backbone.blocks
        for layer in list(layers)[-cfg['partial_layers']:]: layer.requires_grad_(True)
        norm = backbone.ln_enc_f if cfg['spec']['architecture'] == 'encoder_decoder' else (backbone.ln_f if cfg['spec']['architecture'] == 'decoder_only' else backbone.norm_f)
        norm.requires_grad_(True)
    elif method == 'lorma':
        for name, child in list(backbone.named_modules()):
            if isinstance(child, nn.Linear) and '.attn.' in name:
                parent, leaf = name.rsplit('.', 1); setattr(backbone.get_submodule(parent), leaf, ProjectionAdapter(child, cfg['rank'], cfg['alpha'])); targets.append(name)
        assert targets
    elif method == 'ia3':
        for name, child in list(backbone.named_modules()):
            if not isinstance(child, nn.Linear): continue
            output = '.attn.' in name and name.rsplit('.', 1)[-1] in ('k_proj', 'v_proj', 'qkv')
            intermediate = name.endswith('.ff.w2') or name.endswith('.ff.2') or name.endswith('.mlp.2')
            if output or intermediate:
                parent, leaf = name.rsplit('.', 1)
                setattr(backbone.get_submodule(parent), leaf, IA3Projection(child, input_scale=intermediate, fused_kv=name.endswith('.qkv'))); targets.append(name)
        assert targets
    model = LegalRetriever(backbone, cfg)
    metadata = {'phase1_config': phase1, 'phase1_parameters': original_parameters, 'selected_phase1_layers': selected, 'adaptation_targets': targets,
                'total_parameters': sum(p.numel() for p in model.parameters()), 'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad)}
    return model, metadata


def make_chunks(split, role, keys, cfg):
    all_chunks = []; owners = []; coverage = []
    for owner, key in enumerate(keys):
        ids = TOKENS[split][role][key]; size = cfg['chunk_length'] - 2
        starts = list(range(0, max(1, len(ids) - size + 1), size - cfg['chunk_overlap']))
        if ids and starts[-1] + size < len(ids): starts.append(max(0, len(ids) - size))
        starts = list(dict.fromkeys(starts))
        if cfg['max_chunks'] and len(starts) > cfg['max_chunks']:
            count = cfg['max_chunks']; policy = cfg['chunk_policy']
            if policy == 'head': starts = starts[:count]
            elif policy == 'tail': starts = starts[-count:]
            elif policy == 'head_tail': starts = starts[:(count + 1) // 2] + (starts[-(count // 2):] if count // 2 else [])
            else: starts = [starts[i] for i in np.linspace(0, len(starts) - 1, count).round().astype(int)]
        previous = 0; covered = 0
        for start in sorted(starts):
            stop = min(start + size, len(ids)); covered += max(0, stop - max(start, previous)); previous = max(previous, stop)
            all_chunks.append([BOS_ID] + (ids[start:stop] or [UNK_ID]) + [EOS_ID]); owners.append(owner)
        coverage.append(covered / len(ids) if ids else 1.0)
    length = max(map(len, all_chunks)); values = torch.full((len(all_chunks), length), PAD_ID, dtype=torch.long)
    mask = torch.zeros_like(values, dtype=torch.bool); content = torch.zeros_like(mask)
    for i, ids in enumerate(all_chunks):
        values[i, :len(ids)] = torch.tensor(ids); mask[i, :len(ids)] = True; content[i, 1:len(ids) - 1] = True
    return {'ids': values.to(DEVICE), 'mask': mask.to(DEVICE), 'content': content.to(DEVICE),
            'owners': torch.tensor(owners, device=DEVICE), 'keys': keys, 'coverage': coverage}

## 7. Full-pool dense retrieval, chunk late interaction and dev-selected fusion

All candidates are encoded and searched, not just a gold-containing shortlist. Shared neural encoders
produce normalized embeddings. Optional asymmetric heads and chunk late interaction can be compared.
Bounded reciprocal-rank fusion compares dense/late-interaction and BM25 ranks on full dev. Selected
trained-model submissions retain a nonzero neural contribution. Optional exact IndianKanoon URL
features use only the supplied query text and are disabled by default; no web lookups are made.

In [ ]:
@torch.no_grad()
def encode_pool(model, split, role, cfg):
    was_training = model.training; model.eval(); records = QUERIES[split] if role == 'queries' else CANDIDATES[split]
    keys = list(records); vectors = []; chunks = []; owners = []; offset = 0; coverage = []
    for start in tqdm(range(0, len(keys), cfg['encode_batch_size']), desc='Encoding ' + split + '/' + role, unit='batch'):
        selected = keys[start:start + cfg['encode_batch_size']]; batch = make_chunks(split, role, selected, cfg)
        with amp_context(): document, window, owner = model(batch, role, return_chunks=True)
        vectors.append(document.float().cpu()); chunks.append(window.float().cpu()); owners.append(owner.cpu() + offset)
        coverage.extend(batch['coverage']); offset += len(selected)
    model.train(was_training)
    result = {'ids': keys, 'vectors': torch.cat(vectors), 'chunks': torch.cat(chunks), 'owners': torch.cat(owners),
              'mean_token_coverage': float(np.mean(coverage))}
    assert result['vectors'].shape == (len(keys), cfg['embedding_dim'])
    if not torch.isfinite(result['vectors']).all() or not torch.isfinite(result['chunks']).all(): raise FloatingPointError('Nonfinite retrieval embeddings.')
    return result


def citation_signal(text, ids):
    import re
    referenced = set(re.findall(r'(?:indiankanoon\.org/)?doc/(\d+)', text))
    normalized = {str(int(value)) for value in referenced}
    return np.asarray([float(candidate.isdigit() and str(int(candidate)) in normalized) for candidate in ids])


def ranking_components(query_embeddings, candidate_embeddings, split, cfg):
    if query_embeddings['ids'] != list(QUERIES[split]) or candidate_embeddings['ids'] != list(CANDIDATES[split]):
        raise ValueError('Embeddings do not align with this split/pool.')
    ids = candidate_embeddings['ids']; qpositions = {key: i for i, key in enumerate(query_embeddings['ids'])}
    result = {}
    for query in tqdm(QUERIES[split], desc='Ranking ' + split + ' against its own pool', unit='query'):
        i = qpositions[query]
        dense = (query_embeddings['vectors'][i:i + 1] @ candidate_embeddings['vectors'].T)[0].numpy()
        lexical = BM25_RANKS[split][query]
        if len(lexical) < min(cfg['rrf_depth'], len(ids)):
            lexical = stable_order(BM25[split].scores(QUERIES[split][query]['text']), ids)[:cfg['rrf_depth']].astype(np.int32)
            BM25_RANKS[split][query] = lexical
        components = {'dense': stable_order(dense, ids), 'bm25': lexical}
        if cfg['search_late_interaction']:
            q = query_embeddings['chunks'][query_embeddings['owners'] == i]
            similarities = q @ candidate_embeddings['chunks'].T
            maximum = torch.full((len(q), len(ids)), -float('inf'))
            maximum.scatter_reduce_(1, candidate_embeddings['owners'][None].expand(len(q), -1), similarities, reduce='amax', include_self=True)
            late = maximum.mean(0).numpy()
            components['late'] = stable_order(late, ids)
        if cfg['citation_enabled']: components['citation'] = citation_signal(QUERIES[split][query]['text'], ids)
        result[query] = components
    return result


def rank_with_policy(components, split, policy):
    ids = list(CANDIDATES[split]); predictions = {}
    for query, values in components.items():
        mode = policy['neural_mode']; scores = np.zeros(len(ids)); weight = policy['dense_weight']
        # Truncated RRF limits noise from the distant tail; stable candidate-ID ties are deterministic.
        for order, importance in [(values[mode], weight), (values['bm25'], 1 - weight)]:
            count = min(len(order), policy['rrf_depth']); selected = order[:count]
            scores[selected] += importance / (policy['rrf_k'] + np.arange(1, count + 1))
        if policy.get('citation_weight', 0): scores += policy['citation_weight'] * values['citation']
        predictions[query] = top_predictions(scores, ids, query)
    validate_predictions(predictions, split)
    return predictions


def select_policy(components, cfg):
    best = None; trials = []
    modes = ['dense', 'late'] if cfg['search_late_interaction'] else ['dense']
    for mode in modes:
        for weight in cfg['dense_weights']:
            for citation in (cfg['citation_weights'] if cfg['citation_enabled'] else [0.0]):
                policy = {'neural_mode': mode, 'dense_weight': float(weight), 'rrf_k': cfg['rrf_k'], 'rrf_depth': cfg['rrf_depth'], 'citation_weight': float(citation)}
                predictions = rank_with_policy(components, 'dev', policy); metrics = retrieval_metrics(predictions)
                trials.append({'policy': policy, 'micro_f1_at_7': metrics['micro_f1_at_7']})
                # Prefer a larger neural contribution and fewer heuristics when scores tie.
                key = (metrics['micro_f1_at_7'], weight, -citation, mode == 'dense')
                if best is None or key > best[0]: best = (key, policy, predictions, metrics)
    return best[1], best[2], best[3], trials


def dense_predictions(query_embeddings, candidate_embeddings, split):
    ids = candidate_embeddings['ids']; predictions = {}
    for i, query in enumerate(query_embeddings['ids']):
        scores = (query_embeddings['vectors'][i:i + 1] @ candidate_embeddings['vectors'].T)[0].numpy()
        predictions[query] = top_predictions(scores, ids, query)
    validate_predictions(predictions, split)
    return predictions


def evaluate_model(model, cfg, split='dev'):
    queries = encode_pool(model, split, 'queries', cfg); candidates = encode_pool(model, split, 'candidates', cfg)
    components = ranking_components(queries, candidates, split, cfg)
    dense = dense_predictions(queries, candidates, split)
    if split != 'dev': return queries, candidates, components, dense
    policy, predictions, metrics, trials = select_policy(components, cfg)
    return {'queries': queries, 'candidates': candidates, 'components': components,
            'policy': policy, 'predictions': predictions, 'metrics': metrics, 'trials': trials,
            'dense_metrics': retrieval_metrics(dense), 'dense_predictions': dense}


def sample_training_batch(query_keys, cfg, epoch, batch_index, mined=None):
    rng = random.Random(cfg['seed'] + epoch * 1000003 + batch_index * 9743)
    pool = list(CANDIDATES['train']); candidates = []; per_query = {}
    for query in query_keys:
        gold = set(QUERIES['train'][query]['relevant']); positives = sorted(gold)
        selected = rng.sample(positives, min(len(positives), cfg['positives']))
        hard_source = list(dict.fromkeys((mined or {}).get(query, []) + LEXICAL_TOP['train'][query]))
        hard = [key for key in hard_source if key not in gold and (not EXCLUDE_SELF_MATCH or key != query)]
        hard_count = min(len(hard), round(cfg['negatives'] * cfg['hard_fraction']))
        negatives = rng.sample(hard, hard_count)
        available = [key for key in pool if key not in gold and key not in negatives and (not EXCLUDE_SELF_MATCH or key != query)]
        negatives += rng.sample(available, min(len(available), cfg['negatives'] - len(negatives)))
        per_query[query] = set(selected + negatives); candidates.extend(selected + negatives)
    candidates = list(dict.fromkeys(candidates)); positive = []; allowed = []
    for query in query_keys:
        gold = set(QUERIES['train'][query]['relevant'])
        positive.append([key in gold for key in candidates])
        allowed.append([not EXCLUDE_SELF_MATCH or key != query for key in candidates] if cfg['in_batch'] else [key in per_query[query] for key in candidates])
    positive = torch.tensor(positive, dtype=torch.bool, device=DEVICE); allowed = torch.tensor(allowed, dtype=torch.bool, device=DEVICE)
    assert (positive & allowed).any(1).all()
    return candidates, positive, allowed


def retrieval_loss(scores, positive, allowed, cfg):
    logits = (scores / cfg['temperature']).masked_fill(~allowed, -1e4)
    labels = (positive & allowed).float(); labels = labels / labels.sum(1, keepdim=True)
    contrastive = -(labels * F.log_softmax(logits, dim=-1)).sum(1).mean()
    pairwise = []
    for i in range(len(scores)):
        positives = scores[i][positive[i] & allowed[i]]; negatives = scores[i][~positive[i] & allowed[i]]
        if len(negatives): pairwise.append(F.softplus((negatives.max() - positives[:, None] + cfg['margin']) / cfg['temperature']).mean())
    pairwise_loss = torch.stack(pairwise).mean() if pairwise else scores.sum() * 0
    if cfg['loss_name'] == 'multi_positive': return contrastive
    if cfg['loss_name'] == 'pairwise': return pairwise_loss
    return 0.5 * (contrastive + pairwise_loss)


def distillation_loss(scores, teacher_scores, allowed, cfg, teacher_cfg):
    student = (scores / cfg['temperature'] / cfg['kd_temperature']).masked_fill(~allowed, -1e4)
    teacher = (teacher_scores / teacher_cfg['temperature'] / cfg['kd_temperature']).masked_fill(~allowed, -1e4)
    return F.kl_div(F.log_softmax(student, dim=-1), F.softmax(teacher, dim=-1), reduction='batchmean') * cfg['kd_temperature'] ** 2

## 8. Train, select on full dev, reload and verify

Uniform multi-positive contrastive loss, pairwise hardest-negative loss or their combination. In-batch
negatives are masked against every query's full known gold set. Optional train-only dense mining
refreshes hard negatives between epochs and is included in resume state. Query-weighted accumulation
handles short final batches. KD transfers a frozen larger teacher's candidate-score distribution on
train batches, with shared candidate IDs but potentially different chunking/dimensions.

Checkpoints select by full-dev Micro-F1@7 with dense-only Micro-F1 as a tie-breaker. The best model
is reloaded and rankings/metrics must match. JSON is reread and rescored; test IDs/pool membership,
seven-way uniqueness, ranking order and ZIP layout are checked. Test labels are never scored.

In [ ]:
def capture_rng():
    return {'python': random.getstate(), 'numpy': {
        'kind': np.random.get_state()[0], 'keys': np.random.get_state()[1].tolist(),
        'pos': np.random.get_state()[2], 'gauss': np.random.get_state()[3],
        'cached': np.random.get_state()[4]}, 'torch': torch.get_rng_state(),
        'cuda': torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else []}

In [ ]:
def restore_rng(state):
    random.setstate(state['python'])
    n = state['numpy']
    np.random.set_state((n['kind'], np.array(n['keys'], dtype=np.uint32), n['pos'], n['gauss'], n['cached']))
    torch.set_rng_state(state['torch'])
    if DEVICE.type == 'cuda' and state['cuda']:
        torch.cuda.set_rng_state_all(state['cuda'])

In [ ]:
def base_config(name, overrides=None):
    cfg = {'implementation_version': 1, 'experiment': name, 'spec': copy.deepcopy(EXPERIMENTS[name]), 'run_tag': RUN_TAG, 'seed': SEED,
        'epochs': EPOCHS, 'query_batch_size': QUERY_BATCH_SIZE, 'grad_accum': GRAD_ACCUM_STEPS,
        'positives': POSITIVES_PER_QUERY, 'negatives': NEGATIVES_PER_QUERY, 'hard_fraction': HARD_NEGATIVE_FRACTION,
        'in_batch': USE_IN_BATCH_NEGATIVES, 'mine_dense_negatives': MINE_DENSE_NEGATIVES,
        'chunk_length': CHUNK_LENGTH, 'chunk_overlap': CHUNK_OVERLAP, 'max_chunks': MAX_CHUNKS,
        'chunk_batch_size': CHUNK_BATCH_SIZE, 'encode_batch_size': ENCODE_DOCUMENT_BATCH_SIZE,
        'chunk_policy': CHUNK_POLICY, 'pooling': POOLING, 'embedding_dim': EMBEDDING_DIM,
        'dropout': HEAD_DROPOUT, 'share_projection': SHARE_QUERY_CANDIDATE_PROJECTION,
        'loss_name': LOSS_NAME, 'temperature': TEMPERATURE, 'margin': PAIRWISE_MARGIN,
        'head_lr_multiplier': HEAD_LR_MULTIPLIER, 'weight_decay': WEIGHT_DECAY, 'warmup_fraction': WARMUP_FRACTION,
        'patience': EARLY_STOPPING_PATIENCE, 'gradient_checkpointing': GRADIENT_CHECKPOINTING,
        'rank': RANK, 'alpha': ALPHA, 'prompt_tokens': PROMPT_TOKENS, 'partial_layers': PARTIAL_LAYERS,
        'kd_weight': KD_WEIGHT, 'kd_temperature': KD_TEMPERATURE, 'teacher_requested': TEACHER_RUN_DIR,
        'rrf_k': RRF_K, 'rrf_depth': RRF_DEPTH, 'dense_weights': DENSE_WEIGHTS,
        'search_late_interaction': SEARCH_LATE_INTERACTION, 'citation_enabled': ENABLE_CITATION_FEATURE, 'citation_weights': CITATION_WEIGHTS,
        'bm25': {'k1': BM25_K1, 'b': BM25_B, 'max_features': BM25_MAX_FEATURES, 'stop_words': BM25_STOP_WORDS, 'hard_depth': HARD_NEGATIVE_DEPTH},
        'exclude_self_match': EXCLUDE_SELF_MATCH, 'max_optimizer_steps': MAX_OPTIMIZER_STEPS,
        'phase1_heads': PHASE1_HEADS, 'data_fingerprint': DATA_FINGERPRINT, 'tokenizer_hash': TOKENIZER_HASH}
    if LEARNING_RATE is not None: cfg['spec']['lr'] = LEARNING_RATE
    for key, value in (overrides or {}).items():
        if key == 'lr': cfg['spec']['lr'] = value
        elif key in cfg: cfg[key] = value
        else: raise ValueError('Unknown tuning override: ' + key)
    if cfg['bm25'] != LEXICAL_SIGNATURE: raise ValueError('BM25 settings changed; rerun Section 5 before training.')
    if cfg['exclude_self_match'] != DATA_SELF_EXCLUSION: raise ValueError('Self-exclusion changed; rerun Sections 4–5.')
    assert cfg['loss_name'] in ('multi_positive', 'pairwise', 'combined')
    assert cfg['spec']['method'] in ('full', 'partial', 'lorma', 'prompt', 'ia3', 'kd')
    assert 4 <= cfg['chunk_length'] <= 1024 and 0 <= cfg['chunk_overlap'] < cfg['chunk_length'] - 2
    assert cfg['max_chunks'] >= 0 and cfg['chunk_batch_size'] > 0 and cfg['encode_batch_size'] > 0
    assert cfg['positives'] > 0 and cfg['negatives'] > 0 and 0 <= cfg['hard_fraction'] <= 1
    assert cfg['chunk_policy'] in ('spread', 'head', 'tail', 'head_tail') and cfg['pooling'] in ('attention', 'mean', 'max', 'mean_max')
    assert cfg['temperature'] > 0 and cfg['embedding_dim'] > 0 and 0 <= cfg['kd_weight'] <= 1 and cfg['kd_temperature'] > 0
    assert cfg['rrf_k'] > 0 and cfg['rrf_depth'] >= TOP_K + int(EXCLUDE_SELF_MATCH)
    assert cfg['dense_weights'] and all(0 < weight <= 1 for weight in cfg['dense_weights'])
    return cfg


def run_path(name): return PROJECT_DIR / (name + '_' + RUN_TAG)


def resolve_teacher(cfg, student_parameters):
    paths = [Path(cfg['teacher_requested'])] if cfg['teacher_requested'] else [path.parent for path in PROJECT_DIR.glob('*/metrics.json')]
    choices = []
    for path in paths:
        if not (path / 'run_config.json').is_file() or not (path / 'best_model.pt').is_file(): continue
        saved = json.loads((path / 'run_config.json').read_text()); metric = json.loads((path / 'metrics.json').read_text())
        if saved['data_fingerprint'] != DATA_FINGERPRINT or saved['tokenizer_hash'] != TOKENIZER_HASH or saved['phase1_heads'] != PHASE1_HEADS: continue
        if not metric['training_complete'] or metric['total_parameters'] <= student_parameters or saved['spec']['method'] == 'kd': continue
        choices.append((metric['dense_micro_f1_at_7'], str(path)))
    if not choices: raise FileNotFoundError('KD requires a completed compatible larger Phase 7 retrieval teacher. Run encoder_full/decoder_partial first.')
    path = Path(max(choices)[1]); teacher_cfg = json.loads((path / 'run_config.json').read_text())
    model, metadata = make_model(teacher_cfg); state = torch_load(path / 'best_model.pt'); model.load_state_dict(state['model'], strict=True); del state
    stat = (path / 'best_model.pt').stat()
    info = {'path': str(path), 'bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns,
            'teacher_parameters': metadata['total_parameters'], 'selection_metric': 'dense_micro_f1_at_7'}
    return model.to(DEVICE).eval().requires_grad_(False), teacher_cfg, info


def optimizer_groups(model, cfg):
    backbone = []; head = []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad: (backbone if name.startswith('features.backbone.') else head).append(parameter)
    groups = []
    if backbone: groups.append({'params': backbone, 'lr': cfg['spec']['lr']})
    if head: groups.append({'params': head, 'lr': cfg['spec']['lr'] * cfg['head_lr_multiplier']})
    return groups


def run_experiment(name, overrides=None, run_name=None):
    cfg = base_config(name, overrides); run_name = run_name or name; output = run_path(run_name); output.mkdir(parents=True, exist_ok=True)
    config_path = output / 'run_config.json'
    if config_path.is_file() and json.loads(config_path.read_text()) != cfg: raise ValueError('Settings changed under existing run; choose a fresh RUN_TAG: ' + str(output))
    if not RESUME and (output / 'last.pt').exists(): raise ValueError('Use a fresh tag instead of overwriting training.')
    if RESUME and all((output / filename).is_file() for filename in ['metrics.json', 'verification.json', 'dev_prediction.json', 'prediction.json', 'submission.zip', 'best_model.pt', 'last.pt']):
        cached = json.loads((output / 'metrics.json').read_text()); verified = json.loads((output / 'verification.json').read_text())
        if cached['training_complete'] and verified.get('checkpoint_reload_ranks_match') and verified.get('json_metrics_match'):
            predictions = json.loads((output / 'prediction.json').read_text()); validate_predictions(predictions, 'test')
            with zipfile.ZipFile(output / 'submission.zip') as archive:
                assert archive.namelist() == ['prediction.json'] and json.loads(archive.read('prediction.json')) == predictions
            measured = retrieval_metrics(json.loads((output / 'dev_prediction.json').read_text()))
            assert measured['micro_f1_at_7'] == cached['micro_f1_at_7']
            print('Reused completed verified run:', output); return cached
    save_json(cfg, config_path); seed_all(cfg['seed']); model, metadata = make_model(cfg)
    teacher = None; teacher_cfg = None; teacher_info = None
    if cfg['spec']['method'] == 'kd':
        pinned = cfg
        if (output / 'last.pt').is_file() and (output / 'model_metadata.json').is_file():
            old_teacher = json.loads((output / 'model_metadata.json').read_text()).get('teacher')
            if old_teacher: pinned = {**cfg, 'teacher_requested': old_teacher['path']}
        rng = capture_rng()
        try: teacher, teacher_cfg, teacher_info = resolve_teacher(pinned, metadata['total_parameters'])
        finally: restore_rng(rng)
    metadata['teacher'] = teacher_info; save_json(metadata, output / 'model_metadata.json'); model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(optimizer_groups(model, cfg), weight_decay=cfg['weight_decay'])
    batches = math.ceil(len(TRAIN_KEYS) / cfg['query_batch_size']); total_steps = cfg['epochs'] * math.ceil(batches / cfg['grad_accum'])
    warmup = max(1, round(total_steps * cfg['warmup_fraction']))
    def schedule(step): return (step + 1) / warmup if step < warmup else 0.5 * (1 + math.cos(math.pi * min(1, (step - warmup) / max(1, total_steps - warmup))))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda' and AMP_DTYPE == torch.float16)
    epoch0 = 0; batch0 = 0; step = 0; best = (-1., -1.); bad = 0; history = []; complete = False; elapsed = 0.0; mined = {}
    if RESUME and (output / 'last.pt').is_file():
        state = torch_load(output / 'last.pt')
        if state['config'] != cfg or state['teacher'] != teacher_info: raise ValueError('Resume config or teacher signature changed.')
        model.load_state_dict(state['model'], strict=True); optimizer.load_state_dict(state['optimizer']); scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler']); restore_rng(state['rng'])
        epoch0 = state['epoch']; batch0 = state['next_batch']; step = state['step']; best = tuple(state['best']); bad = state['bad']
        history = state['history']; complete = state['complete']; elapsed = state['seconds']; mined = state['mined']; del state
    start = time.time()
    if DEVICE.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    def persist(epoch, next_batch):
        save_torch({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                    'rng': capture_rng(), 'config': cfg, 'teacher': teacher_info, 'epoch': epoch, 'next_batch': next_batch, 'step': step,
                    'best': best, 'bad': bad, 'history': history, 'complete': complete, 'mined': mined, 'seconds': elapsed + time.time() - start}, output / 'last.pt')
        save_json(history, output / 'history.json')
    if not complete:
        for epoch in range(epoch0, cfg['epochs']):
            if cfg['max_optimizer_steps'] is not None and step >= cfg['max_optimizer_steps']: break
            order = list(TRAIN_KEYS); random.Random(cfg['seed'] + epoch).shuffle(order)
            begin = batch0 if epoch == epoch0 else 0; next_batch = begin; capped = False; accumulated = 0.0
            model.train(); optimizer.zero_grad(set_to_none=True)
            progress = tqdm(range(begin, batches), desc=f'{run_name} epoch {epoch + 1}/{cfg["epochs"]}', unit='query batch')
            for i in progress:
                query_keys = order[i * cfg['query_batch_size']:(i + 1) * cfg['query_batch_size']]
                candidates, positive, allowed = sample_training_batch(query_keys, cfg, epoch, i, mined)
                query_batch = make_chunks('train', 'queries', query_keys, cfg); candidate_batch = make_chunks('train', 'candidates', candidates, cfg)
                group_start = (i // cfg['grad_accum']) * cfg['grad_accum']
                group_queries = min(cfg['grad_accum'] * cfg['query_batch_size'], len(order) - group_start * cfg['query_batch_size'])
                fraction = len(query_keys) / group_queries
                with amp_context():
                    q = model(query_batch, 'queries'); c = model(candidate_batch, 'candidates'); scores = q @ c.T
                    loss = retrieval_loss(scores.float(), positive, allowed, cfg)
                    if teacher is not None:
                        with torch.no_grad():
                            tq = teacher(make_chunks('train', 'queries', query_keys, teacher_cfg), 'queries')
                            tc = teacher(make_chunks('train', 'candidates', candidates, teacher_cfg), 'candidates')
                        loss = (1 - cfg['kd_weight']) * loss + cfg['kd_weight'] * distillation_loss(scores.float(), (tq @ tc.T).float(), allowed, cfg, teacher_cfg)
                if not torch.isfinite(loss): raise FloatingPointError('Nonfinite retrieval loss.')
                scaler.scale(loss * fraction).backward(); accumulated += loss.item() * fraction; next_batch = i + 1
                progress.set_postfix(loss=f'{loss.item():.4f}', step=step)
                if next_batch % cfg['grad_accum'] == 0 or next_batch == batches:
                    scaler.unscale_(optimizer); nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                    previous = scaler.get_scale(); scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
                    if scaler.get_scale() >= previous:
                        scheduler.step(); step += 1; history.append({'kind': 'train', 'epoch': epoch + 1, 'step': step, 'loss': accumulated})
                        if step % SAVE_EVERY_STEPS == 0: persist(epoch, next_batch)
                    accumulated = 0.0
                    if cfg['max_optimizer_steps'] is not None and step >= cfg['max_optimizer_steps']: capped = True; break
            progress.close(); evaluated = evaluate_model(model, cfg)
            metrics = evaluated['metrics']; dense_metric = evaluated['dense_metrics']['micro_f1_at_7']; key = (metrics['micro_f1_at_7'], dense_metric)
            history.append({'kind': 'dev', 'epoch': epoch + 1, 'step': step, 'micro_f1_at_7': metrics['micro_f1_at_7'], 'dense_micro_f1_at_7': dense_metric})
            show_scores(metrics, 'Full DEV — dev-selected neural/BM25 policy'); print('Dense-only DEV Micro-F1@7:', dense_metric)
            if key > best:
                best = key; bad = 0
                save_torch({'model': model.state_dict(), 'config': cfg, 'metadata': metadata, 'epoch': epoch + 1}, output / 'best_model.pt')
                save_json(evaluated['policy'], output / 'retrieval_policy.json'); save_json(metrics, output / 'best_dev_metrics.json')
                save_json(evaluated['predictions'], output / 'best_dev_prediction.json')
            else: bad += 1
            finished = next_batch == batches
            complete = finished and (epoch + 1 == cfg['epochs'] or (cfg['patience'] > 0 and bad >= cfg['patience']))
            if cfg['mine_dense_negatives'] and finished and not complete and not capped:
                tq = encode_pool(model, 'train', 'queries', cfg); tc = encode_pool(model, 'train', 'candidates', cfg)
                mined = {}
                for j, query in enumerate(tq['ids']):
                    order_dense = stable_order((tq['vectors'][j:j + 1] @ tc['vectors'].T)[0].numpy(), tc['ids'])
                    mined[query] = [tc['ids'][k] for k in order_dense if tc['ids'][k] not in set(QUERIES['train'][query]['relevant'])][:HARD_NEGATIVE_DEPTH]
            persist(epoch + 1 if finished else epoch, 0 if finished else next_batch)
            del evaluated
            if complete or capped: break
    if teacher is not None: del teacher; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    if not (output / 'best_model.pt').is_file(): raise RuntimeError('No evaluated checkpoint exists.')
    state = torch_load(output / 'best_model.pt'); model.load_state_dict(state['model'], strict=True); best_epoch = state['epoch']; del state
    final = evaluate_model(model, cfg)
    saved_policy = json.loads((output / 'retrieval_policy.json').read_text())
    verified_predictions = rank_with_policy(final['components'], 'dev', saved_policy)
    assert verified_predictions == json.loads((output / 'best_dev_prediction.json').read_text()), 'Reloaded rankings differ.'
    measured = retrieval_metrics(verified_predictions)
    assert measured == json.loads((output / 'best_dev_metrics.json').read_text()), 'Reloaded metrics differ.'
    save_json(verified_predictions, output / 'dev_prediction.json')
    assert retrieval_metrics(json.loads((output / 'dev_prediction.json').read_text())) == measured
    save_json(final['dense_metrics'], output / 'dense_dev_metrics.json'); save_json(final['trials'], output / 'ranking_trials.json')
    save_json(measured['per_query'], output / 'dev_errors.json')
    save_torch({'queries': final['queries'], 'candidates': final['candidates'], 'data_fingerprint': DATA_FINGERPRINT}, output / 'dev_embeddings.pt')
    coverage = {'dev_queries': final['queries']['mean_token_coverage'], 'dev_candidates': final['candidates']['mean_token_coverage']}
    metrics = {**measured, 'dense_micro_f1_at_7': final['dense_metrics']['micro_f1_at_7'], 'bm25_micro_f1_at_7': bm25_metrics['micro_f1_at_7'],
               'experiment': name, 'run_name': run_name, 'architecture': cfg['spec']['architecture'], 'method': cfg['spec']['method'],
               'best_epoch': best_epoch, 'training_complete': complete, 'optimizer_steps': step,
               'training_seconds': elapsed + time.time() - start, 'peak_gpu_memory_bytes': torch.cuda.max_memory_allocated() if DEVICE.type == 'cuda' else 0, **metadata}
    save_json(metrics, output / 'metrics.json'); show_scores(metrics, 'Best checkpoint — independently verified DEV')
    del final
    queries, candidates, components, _ = evaluate_model(model, cfg, 'test')
    predictions = rank_with_policy(components, 'test', saved_policy); save_submission(output, predictions)
    save_torch({'queries': queries, 'candidates': candidates, 'data_fingerprint': DATA_FINGERPRINT}, output / 'test_embeddings.pt')
    coverage.update(test_queries=queries['mean_token_coverage'], test_candidates=candidates['mean_token_coverage'])
    save_json(coverage, output / 'chunk_coverage.json')
    save_json({'checkpoint_reload_ranks_match': True, 'checkpoint_reload_metrics_match': True, 'json_metrics_match': True,
               'split_specific_candidate_pool': True, 'seven_unique_original_candidate_ids': True, 'zip_verified': True,
               'training_complete': complete, 'private_test_score': None}, output / 'verification.json')
    train_rows = [row for row in history if row['kind'] == 'train']; dev_rows = [row for row in history if row['kind'] == 'dev']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4)); axes[0].plot([r['step'] for r in train_rows], [r['loss'] for r in train_rows])
    axes[0].set(xlabel='Optimizer step', ylabel='Retrieval loss'); axes[1].plot([r['epoch'] for r in dev_rows], [r['micro_f1_at_7'] for r in dev_rows], label='Selected hybrid')
    axes[1].plot([r['epoch'] for r in dev_rows], [r['dense_micro_f1_at_7'] for r in dev_rows], label='Dense only'); axes[1].legend()
    axes[1].set(xlabel='Epoch', ylabel='DEV Micro-F1@7'); fig.tight_layout(); fig.savefig(output / 'training_curves.png', dpi=130); plt.show(); plt.close(fig)
    print('Verification PASS. Saved:', output / 'submission.zip')
    del model, optimizer, queries, candidates, components; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return metrics

## 9. Preflight checks

In [ ]:
assert len(FILES)==6 and TOP_K==7
assert all(row['relevant'] is None for row in QUERIES['test'].values())
validate_predictions(BM25_BASELINE['dev'],'dev')
base=nn.Linear(7,12); x=torch.randn(2,3,7)
for adapter in [ProjectionAdapter(copy.deepcopy(base),3,6),IA3Projection(copy.deepcopy(base)),IA3Projection(copy.deepcopy(base),input_scale=True),IA3Projection(copy.deepcopy(base),fused_kv=True)]:
    torch.testing.assert_close(adapter(x),base(x))
    adapter(x).sum().backward()
    assert any(p.grad is not None for p in adapter.parameters() if p.requires_grad)
print('PASS: six files, split pools, test gold isolation, seven-ID schema and LoRMA/IA3 identity/gradients.')

## 10. Run selected/core/extra experiments and optional bounded training trials

In [ ]:
selected=list(CORE_EXPERIMENTS) if RUN_ALL else [EXPERIMENT]
if RUN_EXTRA_APPROACHES: selected=list(dict.fromkeys(selected+list(EXTRA_EXPERIMENTS)))
results=[]
for name in tqdm(selected,desc='Phase 7 experiments',unit='run'):
    results.append(run_experiment(name))
if RUN_TUNING:
    for trial in tqdm(TUNING_TRIALS,desc='Bounded training trials',unit='trial'):
        results.append(run_experiment(TUNE_BASE_EXPERIMENT,trial['overrides'],TUNE_BASE_EXPERIMENT+'__'+trial['name']))

## 11. Experiment comparison and requirement coverage

In [ ]:
import csv
records = []
for file in PROJECT_DIR.glob('*/metrics.json'):
    path = file.parent
    if not (path / 'run_config.json').is_file(): continue
    cfg = json.loads((path / 'run_config.json').read_text())
    if cfg['data_fingerprint'] != DATA_FINGERPRINT or cfg['tokenizer_hash'] != TOKENIZER_HASH: continue
    row = json.loads(file.read_text()); row['path'] = str(path); records.append(row)
complete = [row for row in records if row['training_complete']]
methods = {row['method'] for row in complete}; architectures = {row['architecture'] for row in complete}
digests = []; best_per_origin = {}
for row in sorted(complete, key=lambda r: r['micro_f1_at_7'], reverse=True):
    path = Path(row['path']); predictions = json.loads((path / 'prediction.json').read_text()); validate_predictions(predictions, 'test')
    digests.append(hashlib.sha256(json.dumps(predictions, sort_keys=True).encode()).hexdigest())
    best_per_origin.setdefault(row['architecture'], {'run': row['run_name'], 'method': row['method'],
        'dev_micro_f1_at_7': row['micro_f1_at_7'], 'submission': str(path / 'submission.zip')})
coverage = {'four_completed_runs': len(complete) >= 4, 'six_completed_runs': len(complete) >= 6,
    'six_distinct_prediction_contents': len(set(digests)) >= 6,
    'all_four_phase1_origins': architectures >= set(MODEL_CLASSES),
    'three_non_kd_adaptation_methods': len(methods - {'kd'}) >= 3,
    'four_listed_techniques': len(methods & {'full', 'partial', 'lorma', 'prompt', 'ia3', 'kd'}) >= 4,
    'compulsory_lorma': 'lorma' in methods, 'compulsory_kd': 'kd' in methods,
    'competition_acceptance_verified': False}
save_json(coverage, PROJECT_DIR / 'coverage.json'); save_json(best_per_origin, PROJECT_DIR / 'best_per_architecture.json')
columns = ['run_name', 'architecture', 'method', 'micro_f1_at_7', 'micro_precision_at_7', 'micro_recall_at_7', 'dense_micro_f1_at_7',
           'bm25_micro_f1_at_7', 'best_epoch', 'total_parameters', 'trainable_parameters', 'training_seconds', 'training_complete', 'path']
with (PROJECT_DIR / 'experiment_summary.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=columns); writer.writeheader()
    for row in sorted(records, key=lambda r: r['micro_f1_at_7'], reverse=True):
        writer.writerow({key: row.get(key) for key in columns}); show_scores(row, row['run_name'])
print('Requirement coverage:', coverage)
print('Best per Phase 1 origin:', best_per_origin)

## 12. Colab controls, tuning and interpretation

All editable settings are in **Section 3**. Start with `EXPERIMENT='encoder_full'`, `RUN_ALL=False`.
Run all cells in a new Colab runtime. In a prepared runtime, changing only the experiment means rerunning
Section 3 and Sections 10–11. Rebuild Sections 4–5 if changing preprocessing, self exclusion, lexical
settings, negative depth or the data files. Fresh tags preserve comparisons; identical tags/settings resume.

| Goal | Section 3 control |
|---|---|
| One profile | `EXPERIMENT`, `RUN_ALL=False` |
| Six core profiles | `RUN_ALL=True`; completed matching runs are reused |
| Eight extra profiles | `RUN_EXTRA_APPROACHES=True` |
| Automatic training experiments | `RUN_TUNING=True`, edit `TUNING_TRIALS`, choose `TUNE_BASE_EXPERIMENT` |
| Manual learning rate | `LEARNING_RATE`; None uses each profile's LR |
| Effective query batch | `QUERY_BATCH_SIZE * GRAD_ACCUM_STEPS` |
| Positives/negatives | `POSITIVES_PER_QUERY`, `NEGATIVES_PER_QUERY` |
| BM25 hard negative fraction | `HARD_NEGATIVE_FRACTION`, `HARD_NEGATIVE_DEPTH` |
| In-batch negatives | `USE_IN_BATCH_NEGATIVES`; other queries' positives are checked against each query's complete gold set |
| Dynamic neural hard negatives | `MINE_DENSE_NEGATIVES=True`; additional train-only encoding each epoch |
| Training objective | `LOSS_NAME`: multi_positive/pairwise/combined |
| Score sharpness | `TEMPERATURE`, `PAIRWISE_MARGIN` |
| Document context | `CHUNK_LENGTH`, `CHUNK_OVERLAP`, `MAX_CHUNKS` |
| Chunk selection | `CHUNK_POLICY`: spread/head/tail/head_tail |
| Pooling | `POOLING`: attention/mean/max/mean_max |
| Query/candidate representation | `SHARE_QUERY_CANDIDATE_PROJECTION` |
| Embedding/head | `EMBEDDING_DIM`, `HEAD_DROPOUT`, `HEAD_LR_MULTIPLIER` |
| GPU chunk microbatch | `CHUNK_BATCH_SIZE` |
| Inference document batch | `ENCODE_DOCUMENT_BATCH_SIZE` |
| Adaptation capacity | `RANK`, `ALPHA`, `PROMPT_TOKENS`, `PARTIAL_LAYERS` |
| Distillation | `KD_WEIGHT`, `KD_TEMPERATURE`, `TEACHER_RUN_DIR` |
| BM25 | `BM25_K1`, `BM25_B`, `BM25_MAX_FEATURES`, `BM25_STOP_WORDS` |
| Neural/lexical fusion | `DENSE_WEIGHTS`, `RRF_K`, `RRF_DEPTH`; chosen on dev |
| Chunk late interaction | `SEARCH_LATE_INTERACTION=True`; compares max-per-query-chunk candidate similarity |
| Provided-text citation feature | `ENABLE_CITATION_FEATURE=True`, `CITATION_WEIGHTS`; no external query/API |
| Self candidates | `EXCLUDE_SELF_MATCH`; default false follows the specified candidate pool without adding a rule |
| Duration | `EPOCHS`, `EARLY_STOPPING_PATIENCE` |

**Core profiles:** `encoder_full`, `decoder_partial`, `hybrid_lorma`, `lightweight_prompt`,
`lightweight_ia3`, `lightweight_kd`. Full, partial, LoRMA, prompt and IA³ are genuinely applied to the
backbone; every run also trains its retrieval projection/pooling head. KD removes alternating lightweight
layers from the Phase 1 student and trains it using supervised relevance plus a larger frozen teacher's
similarity distribution on the same train query/candidate batches. It does not use a classifier teacher
from an earlier phase or an externally pretrained embedding model.

**Extra profiles:** `decoder_full`, `encoder_partial`, `encoder_lorma`, `encoder_ia3`, `decoder_ia3`,
`hybrid_full`, `lightweight_full`, `lightweight_lorma`. Hyperparameter variants, pooling changes and
fusion changes do not count as new adaptation techniques. Six core profiles also cover the PDF's
stricter general six-submission instruction, despite §6.5/§13 specifying at least four. Distinct accepted
submissions must still be checked on Codabench; identical prediction contents do not establish six submissions.

For parallel Colab sessions, use separate notebook copies and distinct experiments, set `RUN_ALL=False`,
`RUN_EXTRA_APPROACHES=False`, `RUN_TUNING=False`, and prepare shared token caches first. Do not run the
same experiment/tag concurrently. Run KD after a completed larger Phase 7 teacher exists. The teacher
is chosen by dense-only dev Micro-F1 among eligible larger models, not by how strongly BM25 rescues
its final fusion score. Pin `TEACHER_RUN_DIR` if desired; resumed students retain their saved teacher.

The twelve automatic trials really train their overrides in separate `<base>__<trial>_<tag>` folders.
They are independent one-factor changes, not cumulative. Shorten the list according to measured runtime.
Keep official split identities fixed. No training queries use dev/test relevance labels; test labels are
ignored. All ranking components search the complete corresponding candidate pool. Chunk selection
is sampled context, not complete document reading; inspect `chunk_coverage.json`.

The multi-positive loss distributes target probability uniformly across every known positive present
in that sampled candidate batch. Hard negatives exclude the complete gold set; in-batch negatives are
also checked against it. This avoids teaching a known positive as negative merely because it belongs to
another query's sampled set. Pairwise loss compares positives to the hardest sampled negative. Queries
with no labeled positives remain in preprocessing/audits but cannot form positive-supervised train batches.

Default validation compares dense cosine retrieval, optional chunk late interaction, and a bounded
nonzero-neural/BM25 reciprocal-rank fusion search. The standalone BM25 baseline is separately saved;
pure BM25 is not selected as an allegedly adapted neural experiment. If it beats every learned run,
report that result and investigate training/representations. Ties in checkpoint selection use dense-only
dev F1, so a flat hybrid score does not hide improvements in the learned representation.

**Metrics:** Micro-F1@7 = `2*total_hits/(7*query_count + complete_gold_count)`.
Precision = `total_hits/(7*query_count)`; recall = `total_hits/complete_gold_count`.
Gold sets of fewer/more than seven are neither padded nor truncated. Empty dev gold sets still contribute
seven predictions to the precision/F1 denominator. The oracle score reports the maximum possible F1
under the exact-seven rule. Keep `TOP_K=7`; changing it violates the submission contract.

Numeric-only gold IDs resolve to zero-padded candidate IDs only when the exact ID is missing and the
numeric match is unambiguous. Original pool strings are always written. Test predictions are objects,
not the Phase 6 summary list format. Every test query must have seven unique strings from test_candidates.
Duplicate IDs with conflicting case contents fail instead of being overwritten. Empty text is audited and
encoded with UNK. A pool with no usable text stops for schema inspection.

Outputs: `MyDrive/cs781_phase7/<experiment>_<RUN_TAG>/`. Keep best/last checkpoints for inference/resume,
dev/test embeddings for ranking analysis, full configs/metrics, error reports and submission ZIPs.
The root CSV and `best_per_architecture.json` compare complete compatible runs. `bm25_baseline/` is an
extra diagnostic submission, not a required adaptation method. No private test score is fabricated and
the code does not upload to Codabench. Inspect missed relevant cases, query length/coverage and lexical
versus neural rankings before drawing legal-relevance conclusions. Repeated dev tuning may overfit.
Keep the required LLM declaration and references in your report.

References: local capstone PDF §13, professor's Phase 1 starter workflow, your Phase 1 architecture
definitions and strict loading, previous phase Drive/RNG handling. No previous phase's task classifier
is used as pretrained retrieval weights.

- [BM25 reference](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html).
- [IA³ reference implementation description](https://github.com/huggingface/peft/blob/main/docs/source/package_reference/ia3.md).
- [LoRMA](https://aclanthology.org/2025.findings-acl.527/), [prompt tuning](https://arxiv.org/abs/2104.08691).

BM25 uses a nonnegative log1p IDF variant and presence-only query terms. All settings are saved.
Keep the required LLM declaration and base conclusions on actual runs. No leaderboard score is promised.